In [1]:
## samantads.com

import pandas as pd
import numpy as np
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
import pingouin as pg
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import scipy as sp
import sympy as sy
import warnings

warnings.filterwarnings("ignore")

#%% Criando o banco de dados do exemplo

dados = pd.DataFrame({
    "Observacao": [1, 2, 3, 4, 5, 6, 7, 8],
    "Atendimento": [8, 7, 9, 6, 7, 5, 4, 6],
    "Preco":       [5, 7, 4, 6, 6, 8, 9, 7],
    "Qualidade":   [9, 8, 9, 7, 8, 6, 5, 7]
})

print(dados)

#%% Visualizando informações gerais do dataset

print(dados.info())

# Estatísticas descritivas das variáveis numéricas

print(dados.describe())

#%% Selecionando somente as variáveis usadas no PCA

dados_pca = dados[["Atendimento", "Preco", "Qualidade"]]

#%% Matriz de correlações de Pearson

# A matriz de correlação mostra como as variáveis se relacionam entre si

pg.rcorr(
    dados_pca,
    method='pearson',
    upper='pval',
    decimals=4,
    pval_stars={0.01: '***', 0.05: '**', 0.10: '*'}
)

#%% Outra forma de visualizar a matriz de correlações

corr = dados_pca.corr()

print(corr)

#%% Heatmap da matriz de correlações

plt.figure(figsize=(8, 6), dpi=300)

sns.heatmap(
    corr,
    cmap=plt.cm.coolwarm,
    vmax=1,
    vmin=-1,
    center=0,
    square=True,
    linewidths=0.5,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 12},
    cbar_kws={"shrink": 0.70}
)

plt.title('Matriz de Correlações de Pearson', fontsize=14)
plt.tight_layout()
plt.show()

#%% Teste de Esfericidade de Bartlett

# O teste verifica se a matriz de correlações é diferente da matriz identidade.
# Se o p-valor for menor que 0,05, há indícios de correlação suficiente
# entre as variáveis para prosseguir com o PCA.

bartlett, p_value = calculate_bartlett_sphericity(dados_pca)

print(f'Qui² Bartlett: {round(bartlett, 4)}')
print(f'p-valor: {round(p_value, 6)}')

#%% Definindo a PCA com todos os fatores possíveis

# Como temos 3 variáveis, o número máximo de fatores/componentes é 3

fa = FactorAnalyzer(
    n_factors=3,
    method='principal',
    rotation=None
).fit(dados_pca)

#%% Obtendo os autovalores

# Os autovalores indicam a quantidade de variância explicada por cada fator

autovalores = fa.get_eigenvalues()[0]

print("Autovalores:")
print(autovalores)

print("\nSoma dos autovalores:")
print(round(autovalores.sum(), 4))

#%% Obtendo autovalores e autovetores de forma didática

# Esta etapa é apenas para mostrar o fundamento matemático do PCA.
# O FactorAnalyzer já faz esse processo internamente.

lamda = sy.symbols('lamda')
sy.init_printing(scale=0.8)

matriz_corr = sy.Matrix(corr)
polinomio = matriz_corr.charpoly(lamda)

print("Polinômio característico:")
print(polinomio.as_expr())

# Autovalores e autovetores pela decomposição da matriz de correlação

autovalores_np, autovetores_np = sp.linalg.eigh(corr)

# Ordenando do maior para o menor autovalor

autovalores_np = autovalores_np[::-1]
autovetores_np = autovetores_np[:, ::-1]

tabela_autovetores = pd.DataFrame(
    autovetores_np,
    index=dados_pca.columns,
    columns=[f"Autovetor {i+1}" for i in range(len(autovalores_np))]
)

print("\nAutovalores ordenados:")
print(autovalores_np)

print("\nAutovetores:")
print(tabela_autovetores)

#%% Tabela com autovalores, variância explicada e variância acumulada

autovalores_fatores = fa.get_factor_variance()

tabela_eigen = pd.DataFrame(autovalores_fatores)

tabela_eigen.columns = [f"Fator {i+1}" for i in range(tabela_eigen.shape[1])]
tabela_eigen.index = ["Autovalor", "Variância", "Variância Acumulada"]
tabela_eigen = tabela_eigen.T

print(tabela_eigen)

#%% Gráfico da variância explicada por fator

plt.figure(figsize=(8, 6), dpi=300)

ax = sns.barplot(
    x=tabela_eigen.index,
    y='Variância',
    hue=tabela_eigen.index,
    data=tabela_eigen,
    palette='viridis',
    legend=False
)

for container in ax.containers:
    labels = [f"{v*100:.2f}%" for v in container.datavalues]
    ax.bar_label(container, labels=labels)

ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, pos: f"{x*100:.0f}%"))

plt.title("Variância Explicada pelos Fatores", fontsize=14)
plt.xlabel("Fatores extraídos", fontsize=12)
plt.ylabel("Percentual de variância explicada", fontsize=12)
plt.tight_layout()
plt.show()

#%% Determinando as cargas fatoriais

# As cargas fatoriais mostram a relação de cada variável com cada fator

cargas_fatoriais = fa.loadings_

tabela_cargas = pd.DataFrame(
    cargas_fatoriais,
    index=dados_pca.columns,
    columns=[f"Fator {i+1}" for i in range(cargas_fatoriais.shape[1])]
)

print(tabela_cargas)

#%% Loading plot dos dois primeiros fatores

# O gráfico ajuda a visualizar quais variáveis estão mais associadas
# aos dois primeiros fatores

plt.figure(figsize=(8, 6), dpi=300)

tabela_cargas_chart = tabela_cargas.reset_index()
tabela_cargas_chart = tabela_cargas_chart.rename(columns={"index": "Variavel"})

plt.scatter(
    tabela_cargas_chart["Fator 1"],
    tabela_cargas_chart["Fator 2"],
    s=70
)

for i, row in tabela_cargas_chart.iterrows():
    plt.text(
        row["Fator 1"] + 0.03,
        row["Fator 2"],
        row["Variavel"],
        fontsize=10
    )

plt.axhline(y=0, color='grey', linestyle='--')
plt.axvline(x=0, color='grey', linestyle='--')

plt.xlim([-1.2, 1.2])
plt.ylim([-1.2, 1.2])

plt.title("Loading Plot", fontsize=14)
plt.xlabel(f"Fator 1: {round(tabela_eigen.iloc[0]['Variância']*100, 2)}% da variância", fontsize=12)
plt.ylabel(f"Fator 2: {round(tabela_eigen.iloc[1]['Variância']*100, 2)}% da variância", fontsize=12)

plt.tight_layout()
plt.show()

#%% Determinando as comunalidades

# As comunalidades indicam quanto da variância de cada variável
# é representada pelos fatores extraídos

comunalidades = fa.get_communalities()

tabela_comunalidades = pd.DataFrame(
    comunalidades,
    index=dados_pca.columns,
    columns=["Comunalidades"]
)

print(tabela_comunalidades)

#%% Extraindo os fatores para cada observação

# Cada observação passa a receber uma pontuação em cada fator

fatores = pd.DataFrame(fa.transform(dados_pca))
fatores.columns = [f"Fator {i+1}" for i in range(fatores.shape[1])]

dados_com_fatores = pd.concat(
    [dados.reset_index(drop=True), fatores],
    axis=1
)

print(dados_com_fatores)

#%% Identificando os scores fatoriais

# Os scores fatoriais são os pesos usados para formar cada fator
# a partir das variáveis padronizadas

scores = fa.weights_

tabela_scores = pd.DataFrame(
    scores,
    index=dados_pca.columns,
    columns=[f"Fator {i+1}" for i in range(scores.shape[1])]
)

print(tabela_scores)

#%% Conferindo se os fatores são ortogonais

# Quando não há rotação oblíqua, os fatores/componentes tendem a ser ortogonais,
# ou seja, sem correlação entre si.

pg.rcorr(
    dados_com_fatores[["Fator 1", "Fator 2", "Fator 3"]],
    method='pearson',
    upper='pval',
    decimals=4,
    pval_stars={0.01: '***', 0.05: '**', 0.10: '*'}
)

#%% Critério de Kaiser

# Pelo critério de Kaiser, mantemos os fatores com autovalor maior que 1.

fatores_kaiser = tabela_eigen[tabela_eigen["Autovalor"] > 1]

print("Fatores mantidos pelo critério de Kaiser:")
print(fatores_kaiser)

#%% Reestimando a PCA somente com os fatores selecionados

# Neste exemplo, a quantidade de fatores mantidos depende dos autovalores > 1.

n_fatores_kaiser = fatores_kaiser.shape[0]

fa_kaiser = FactorAnalyzer(
    n_factors=n_fatores_kaiser,
    method='principal',
    rotation=None
).fit(dados_pca)

#%% Autovalores, variância explicada e acumulada dos fatores selecionados

autovalores_fatores_kaiser = fa_kaiser.get_factor_variance()

tabela_eigen_kaiser = pd.DataFrame(autovalores_fatores_kaiser)

tabela_eigen_kaiser.columns = [f"Fator {i+1}" for i in range(tabela_eigen_kaiser.shape[1])]
tabela_eigen_kaiser.index = ["Autovalor", "Variância", "Variância Acumulada"]
tabela_eigen_kaiser = tabela_eigen_kaiser.T

print(tabela_eigen_kaiser)

#%% Cargas fatoriais dos fatores selecionados

cargas_fatoriais_kaiser = fa_kaiser.loadings_

tabela_cargas_kaiser = pd.DataFrame(
    cargas_fatoriais_kaiser,
    index=dados_pca.columns,
    columns=[f"Fator {i+1}" for i in range(cargas_fatoriais_kaiser.shape[1])]
)

print(tabela_cargas_kaiser)

#%% Comunalidades considerando apenas os fatores selecionados

comunalidades_kaiser = fa_kaiser.get_communalities()

tabela_comunalidades_kaiser = pd.DataFrame(
    comunalidades_kaiser,
    index=dados_pca.columns,
    columns=["Comunalidades"]
)

print(tabela_comunalidades_kaiser)

#%% Extraindo os fatores finais para as observações

fatores_finais = pd.DataFrame(fa_kaiser.transform(dados_pca))
fatores_finais.columns = [f"Fator {i+1}" for i in range(fatores_finais.shape[1])]

dados_final = pd.concat(
    [dados.reset_index(drop=True), fatores_finais],
    axis=1
)

print(dados_final)

#%% Scores fatoriais finais

scores_finais = fa_kaiser.weights_

tabela_scores_finais = pd.DataFrame(
    scores_finais,
    index=dados_pca.columns,
    columns=[f"Fator {i+1}" for i in range(scores_finais.shape[1])]
)

print(tabela_scores_finais)

#%% Criando um ranking com base nos fatores selecionados

# O ranking é calculado pela soma ponderada dos fatores,
# usando a variância explicada como peso.

dados_final["Ranking"] = 0

for fator in tabela_eigen_kaiser.index:
    peso_variancia = tabela_eigen_kaiser.loc[fator, "Variância"]
    dados_final["Ranking"] = dados_final["Ranking"] + dados_final[fator] * peso_variancia

# Ordenando do maior para o menor ranking

dados_ranking = dados_final.sort_values("Ranking", ascending=False)

print(dados_ranking)

#%% Fim da análise

ModuleNotFoundError: No module named 'factor_analyzer'